Extract om2 data for autoencoder tutorial example. see here https://github.com/PaulSpence/OM2-emulator/issues/7

In [2]:
import os
import re
import sys
import glob
import time
import yaml
import intake
import warnings
import numpy as np
import pandas as pd
import scipy as io
import xarray as xr
import matplotlib.dates as mdates

from mpi4py import MPI
from datetime import date, datetime, timedelta
from matplotlib import pyplot as plt
from dask.distributed import Client
from collections import defaultdict
from xarray.coding.times import CFDatetimeCoder

import gsw as gsw
#convert from psu to abs salinity

warnings.filterwarnings("ignore") # Suppress warnings for these docs
time_coder = CFDatetimeCoder(use_cftime=True)

In [3]:
client = Client()
client


Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /proxy/8787/status,
Dashboard: /proxy/8787/status,Workers: 1
Total threads: 1,Total memory: 251.19 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:45983,Workers: 0
Dashboard: /proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35931,Total threads: 1
Dashboard: /proxy/46393/status,Memory: 251.19 GiB
Nanny: tcp://127.0.0.1:42789,


In [4]:
#https://github.com/COSIMA/cosima-recipes/blob/main/01-Cooking-Lessons-101/01-Basics/01-Loading-Slicing-Dicing-Output.ipynb
catalog = intake.cat.access_nri

In [5]:
catalog.search(name='1deg_jra55_iaf_omip2_cycle6')

,model,description,realm,frequency,variable
name,,,,,
1deg_jra55_iaf_omip2_cycle6,{ACCESS-OM2},{Cycle 6/6 of 1 degree ACCESS-OM2-BGC global configuration with JRA55-do v1.4 OMIP2 interannual forcing (1958-2018)},"{seaIce, ocean}","{fx, 1yr, 1day, 1mon}","{total_ocean_hflux_prec, fsens_ai_m, ty_trans_rho, psiu, temp_yflux_submeso_int_z, surface_pot_temp_max, sens_heat, bmf_v, total_ocean_runoff_heat, strocnx_m, uvel, st_edges_ocean, tx_trans_gm, to..."


In [6]:
variables = catalog.search(name="1deg_jra55_iaf_omip2_cycle6").unique().variable
print(variables)

['total_ocean_hflux_prec', 'fsens_ai_m', 'ty_trans_rho', 'temp_yflux_submeso_int_z', 'psiu', 'surface_pot_temp_max', 'sens_heat', 'bmf_v', 'total_ocean_runoff_heat', 'strocnx_m', 'uvel', 'st_edges_ocean', 'total_ocean_lw_heat', 'tx_trans_gm', 'usq', 'sss_sq', 'sss', 'sfc_hflux_from_water_prec', 'frzmlt', 'pco2', 'vsurf', 'flat_ai_m', 'congel', 'temp_rivermix', 'daidtd_m', 'temp_xflux_ndiffuse_int_z', 'total_ocean_salt', 'xu_ocean', 'tarea', 'paco2', 'fmeltt_ai_m', 'ULAT', 'salt_eta_smooth', 'snoice', 'dyt', 'NCAT', 'salt_global_ave', 'sfc_salt_flux_runoff', 'albsno_m', 'temp_xflux_sigma', 'evap_ai_m', 'rhoave', 'temp_int_rhodz', 'sst_sq', 'yu_ocean', 'bottom_temp', 'fprec', 'total_ocean_sens_heat', 'mld_max', 'time', 'average_DT', 'total_ocean_fprec', 'ANGLE', 'st_ocean', 'salt_nonlocal_KPP', 'divu_m', 'fcondtop_ai_m', 'geolon_t', 'fcondtopn_ai_m', 'albice_m', 'grid_xu_ocean', 'fgo2_raw', 'ice_present_m', 'flwdn_m', 'frazil_3d_int_z', 'daidtt_m', 'bih_fric_u', 'agm', 'stf07', 'surface_

In [7]:
experiment = "1deg_jra55_iaf_omip2_cycle6"
variables = ["net_sfc_heating","frazil_3d_int_z","temp","rho_dzt","rho"]

ds = catalog[experiment].search(frequency="1mon",variable=variables).to_dask(xarray_open_kwargs = dict(use_cftime=True))

In [8]:
ds=ds.sel(time=slice('2000-01-01', '2018-12-31'))
ds

<xarray.Dataset> Size: 15GB
Dimensions:          (time: 228, yt_ocean: 300, xt_ocean: 360, st_ocean: 50)
Coordinates:
  * time             (time) object 2kB 2000-01-14 12:00:00 ... 2018-12-14 12:...
  * yt_ocean         (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 89.32 89.77
  * xt_ocean         (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 78.5 79.5
  * st_ocean         (st_ocean) float64 400B 1.152 3.649 ... 5.034e+03 5.254e+03
Data variables:
    frazil_3d_int_z  (time, yt_ocean, xt_ocean) float32 98MB dask.array<chunksize=(1, 300, 360), meta=np.ndarray>
    net_sfc_heating  (time, yt_ocean, xt_ocean) float32 98MB dask.array<chunksize=(1, 300, 360), meta=np.ndarray>
    temp             (time, st_ocean, yt_ocean, xt_ocean) float32 5GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
    rho_dzt          (time, st_ocean, yt_ocean, xt_ocean) float32 5GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
    rho              (time, st_ocean, yt_ocean, xt_ocean) float32 5GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['frazil_3d_int_z', 'net_sfc_he...
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [21]:
area_t = catalog[experiment].search(variable="area_t").to_dask(progressbar=False)
area_t=area_t["area_t"]
area_t

<xarray.DataArray 'area_t' (yt_ocean: 300, xt_ocean: 360)> Size: 432kB
dask.array<open_dataset-area_t, shape=(300, 360), dtype=float32, chunksize=(300, 360), chunktype=numpy.ndarray>
Coordinates:
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5
    geolat_t  (yt_ocean, xt_ocean) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
Attributes:
    long_name:     tracer cell area
    units:         m^2
    valid_range:   [0.e+00 1.e+15]
    cell_methods:  time: point

In [10]:
total_surface_heat_flx=ds["net_sfc_heating"]+ds["frazil_3d_int_z"]

In [11]:
total_surface_heat_flx

<xarray.DataArray (time: 228, yt_ocean: 300, xt_ocean: 360)> Size: 98MB
dask.array<add, shape=(228, 300, 360), dtype=float32, chunksize=(1, 300, 360), chunktype=numpy.ndarray>
Coordinates:
  * time      (time) object 2kB 2000-01-14 12:00:00 ... 2018-12-14 12:00:00
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5

In [12]:
total_surface_heat_flx = total_surface_heat_flx.assign_attrs(
    long_name="Total surface heat flux including frazil ice from 1deg_jra55_iaf_omip2_cycle6",
    units=ds["net_sfc_heating"].attrs.get("units", "W m-2"),
    description="net_sfc_heating + vertically integrated frazil heat flux"
)

total_surface_heat_flx.name = "total_surface_heat_flx"
total_surface_heat_flx.to_netcdf("1deg_total_surface_heat_flx.nc")

In [13]:
dzt=ds["rho_dzt"]/ds["rho"]
dzt[1,:,100,1].values

array([  2.3341784,   2.7261283,   3.1838586,   3.7184029,   4.342601 ,
         5.0712967,   5.9208345,   6.9108543,   8.067094 ,   9.416018 ,
        10.988602 ,  12.821363 ,  14.95263  ,  17.436405 ,  20.332586 ,
        23.699686 ,  27.61026  ,  32.139927 ,  37.37433  ,  43.396942 ,
        50.29235  ,  58.134373 ,  66.97517  ,  76.828125 ,  87.652115 ,
        99.33197  , 111.664    , 124.35859  , 137.0537   , 149.35133  ,
       160.87875  , 171.3354   , 180.52141  , 188.35287  , 194.85257  ,
       200.12001  , 204.3023   , 207.56195  , 210.05708  , 211.93814  ,
       213.33392  , 214.3505   , 215.07356  , 215.57373  , 215.9028   ,
       216.10297  , 216.20465  , 216.23267  ,         nan,         nan],
      dtype=float32)

In [14]:
cp=3992.10322329649
rho0=1035

In [16]:
ocean_heat=(ds["temp"]*dzt).sum("st_ocean")*cp*rho0
ocean_heat=ocean_heat.load()

In [17]:
ocean_heat = ocean_heat.assign_attrs(
    long_name="Vertically integrated ocean heat content from 1deg_jra55_iaf_omip2_cycle6",
    units="J/m2",
    description="((temp*rho_dzt/dzt).sum(st_ocean)*cp*rho0)"
)

ocean_heat.name = "ocean_heat_content_2d"
ocean_heat.to_netcdf("/home/561/pas561/jk72pas561/jnb/PyEarthTools/notebooks/tutorial/1deg_ocean_heat_content_2d.nc")

In [22]:
xr.Dataset(
    {
        "area_t": area_t,
        "total_surface_heat_flx": total_surface_heat_flx,
        "ocean_heat_content_2d": ocean_heat,
    }
).to_netcdf("/home/561/pas561/jk72pas561/jnb/PyEarthTools/notebooks/tutorial/1deg_ocean_heat_emulator_data.nc")


In [ ]:
#https://access-nri-intake-catalog.readthedocs.io/en/latest/usage/quickstart.html
#From the OMIP-2 cycle6 run: /g/data/ik11/outputs/access-om2/1deg_jra55_iaf_omip2_cycle6
catalog = intake.cat.access_nri
esm_datastore = catalog.search(name="1deg_jra55_iaf_omip2_cycle6").to_source()

#esm_datastore

In [8]:
#esm_datastore.keys()

['ocean.1day.nv:2.xt_ocean:360.xu_ocean:360.yt_ocean:300.yu_ocean:300.max,mean,min',
 'ocean.1day.nv:2.xt_ocean:360.yt_ocean:300.mean',
 'ocean.1day.scalar_axis:1.point',
 'ocean.1mon.grid_xt_ocean:360.grid_xu_ocean:360.grid_yt_ocean:300.grid_yu_ocean:300.neutral:80.neutralrho_edges:81.nv:2.potrho:80.potrho_edges:81.st_edges_ocean:51.st_ocean:50.sw_edges_ocean:51.sw_ocean:50.xt_ocean:360.xu_ocean:360.yt_ocean:300.yu_ocean:300.max,mean,mean_pow(02),min',
 'ocean.1mon.nv:2.scalar_axis:1.mean',
 'ocean.1mon.nv:2.st_edges_ocean:51.st_ocean:50.xt_ocean:360.yt_ocean:300.mean',
 'ocean.1yr.nv:2.st_edges_ocean:51.st_ocean:50.xt_ocean:360.xu_ocean:360.yt_ocean:300.yu_ocean:300.mean',
 'ocean.1yr.nv:2.st_edges_ocean:51.st_ocean:50.xt_ocean:360.yt_ocean:300.mean',
 'ocean.fx.xt_ocean:360.xu_ocean:360.yt_ocean:300.yu_ocean:300.point',
 'seaIce.1day.d2:2.nc:5.ni:360.nj:300.mean',
 'seaIce.1mon.d2:2.nc:5.ni:360.nj:300.mean']

In [ ]:
#esm_datastore.df.head()

In [ ]:
#esm_datastore.search(frequency='fx').interactive

In [10]:
net_sfc_heat = esm_datastore.search(frequency="1mon", variable="net_sfc_heating").to_dask(progressbar=False)

In [11]:
net_sfc_heat

<xarray.Dataset> Size: 316MB
Dimensions:          (time: 732, yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * time             (time) datetime64[ns] 6kB 1958-01-14T12:00:00 ... 2018-1...
  * yt_ocean         (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 89.32 89.77
  * xt_ocean         (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 78.5 79.5
Data variables:
    net_sfc_heating  (time, yt_ocean, xt_ocean) float32 316MB dask.array<chunksize=(1, 300, 360), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['net_sfc_heating']
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [12]:
frazil = esm_datastore.search(frequency="1mon", variable="frazil_3d_int_z").to_dask(progressbar=False)
frazil

<xarray.Dataset> Size: 316MB
Dimensions:          (time: 732, yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * time             (time) datetime64[ns] 6kB 1958-01-14T12:00:00 ... 2018-1...
  * yt_ocean         (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 89.32 89.77
  * xt_ocean         (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 78.5 79.5
Data variables:
    frazil_3d_int_z  (time, yt_ocean, xt_ocean) float32 316MB dask.array<chunksize=(1, 300, 360), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['frazil_3d_int_z']
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [16]:
area_t = esm_datastore.search(variable="area_t").to_dask(progressbar=False)
area_t

<xarray.Dataset> Size: 1MB
Dimensions:   (yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5
    geolat_t  (yt_ocean, xt_ocean) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
    geolon_t  (yt_ocean, xt_ocean) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
Data variables:
    area_t    (yt_ocean, xt_ocean) float32 432kB dask.array<chunksize=(300, 360), meta=np.ndarray>
Attributes: (12/20)
    filename:                                 ocean_grid.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['area_t']
    intake_esm_attrs:filename:                ocean_grid.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: point,time: point,time: p...
    intake_esm_attrs:variable_units:          m^2,m^2,dimensionless,m,m,m,m,d...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          point
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.fx.xt_ocean:360.xu_ocean:...

In [17]:
temp = esm_datastore.search(variable="temp").to_dask(progressbar=False)
temp

<xarray.Dataset> Size: 16GB
Dimensions:   (time: 732, st_ocean: 50, yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * time      (time) datetime64[ns] 6kB 1958-01-14T12:00:00 ... 2018-12-14T12...
  * st_ocean  (st_ocean) float64 400B 1.152 3.649 6.565 ... 5.034e+03 5.254e+03
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5
Data variables:
    temp      (time, st_ocean, yt_ocean, xt_ocean) float32 16GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['temp']
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [19]:
rho_dzt = esm_datastore.search(variable="rho_dzt").to_dask(progressbar=False)
rho_dzt

<xarray.Dataset> Size: 16GB
Dimensions:   (time: 732, st_ocean: 50, yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * time      (time) datetime64[ns] 6kB 1958-01-14T12:00:00 ... 2018-12-14T12...
  * st_ocean  (st_ocean) float64 400B 1.152 3.649 6.565 ... 5.034e+03 5.254e+03
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5
Data variables:
    rho_dzt   (time, st_ocean, yt_ocean, xt_ocean) float32 16GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['rho_dzt']
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [20]:
rho = esm_datastore.search(variable="rho").to_dask(progressbar=False)
rho

<xarray.Dataset> Size: 16GB
Dimensions:   (time: 732, st_ocean: 50, yt_ocean: 300, xt_ocean: 360)
Coordinates:
  * time      (time) datetime64[ns] 6kB 1958-01-14T12:00:00 ... 2018-12-14T12...
  * st_ocean  (st_ocean) float64 400B 1.152 3.649 6.565 ... 5.034e+03 5.254e+03
  * yt_ocean  (yt_ocean) float64 2kB -77.88 -77.63 -77.38 ... 88.87 89.32 89.77
  * xt_ocean  (xt_ocean) float64 3kB -279.5 -278.5 -277.5 ... 77.5 78.5 79.5
Data variables:
    rho       (time, st_ocean, yt_ocean, xt_ocean) float32 16GB dask.array<chunksize=(1, 25, 150, 180), meta=np.ndarray>
Attributes: (12/17)
    filename:                                 ocean_month.nc
    title:                                    ACCESS-OM2-BGC
    grid_type:                                mosaic
    grid_tile:                                1
    intake_esm_vars:                          ['rho']
    intake_esm_attrs:filename:                ocean_month.nc
    ...                                       ...
    intake_esm_attrs:variable_cell_methods:   time: mean,,,,time: mean,time: ...
    intake_esm_attrs:variable_units:          yr,days,days since 0001-01-01 0...
    intake_esm_attrs:realm:                   ocean
    intake_esm_attrs:temporal_label:          max,mean,mean_pow(02),min
    intake_esm_attrs:_data_format_:           netcdf
    intake_esm_dataset_key:                   ocean.1mon.grid_xt_ocean:360.gr...

In [13]:
esm_datastore_filtered = esm_datastore.search(variable="net_sfc_heating")
esm_datastore_filtered.keys()

['ocean.1mon.grid_xt_ocean:360.grid_xu_ocean:360.grid_yt_ocean:300.grid_yu_ocean:300.neutral:80.neutralrho_edges:81.nv:2.potrho:80.potrho_edges:81.st_edges_ocean:51.st_ocean:50.sw_edges_ocean:51.sw_ocean:50.xt_ocean:360.xu_ocean:360.yt_ocean:300.yu_ocean:300.max,mean,mean_pow(02),min']

In [ ]:
dataset = esm_datastore_filtered.search(frequency="1mon").to_dask(progressbar=False)

dataset

In [5]:
print('Starting ACCESS-OM2 and SOTS T/S analysis.')
sys.stdout.flush()

catalog = intake.cat.access_nri
#print(catalog.keys())

#----------------------------------------------------------------------------------------------
# Date of interest 
#----------------------------------------------------------------------------------------------

start_date = np.datetime64("2010-01-01")
end_date = np.datetime64("2020-12-31")

#----------------------------------------------------------------------------------------------
# Variable and degrees of interest in ACCESS-OM2 
#----------------------------------------------------------------------------------------------

experiments = ['1deg_jra55_iaf_omip2_cycle6'] #['01deg_jra55v140_iaf_cycle4', '01deg_jra55v140_iaf_cycle4_jra55v150_extension']]
#experiments = ['1deg_jra55_iaf_omip2_cycle6', ['01deg_jra55v140_iaf_cycle4', '01deg_jra55v140_iaf_cycle4_jra55v150_extension']]

#with bgc 025deg_jra55_iaf_omip2_cycle7_bgc 
#----------------------------------------------------------------------------------------------
# Variables of interest
#----------------------------------------------------------------------------------------------
#net_sfc_heating = sfc_hflux_coupler + sfc_hflux_pme + sfc_hflux_from_runoff + sfc_hflux_from_calving

#sfc_hflux_coupler = swflx + lw_heat + fprec_melt_heat + calving_melt_heat + sens_heat + evap_heat + mh_flux + liceht


var  =  ('pot_temp','salt')

#----------------------------------------------------------------------------------------------
# Region of interest
#----------------------------------------------------------------------------------------------

#region_lat   = np.array([-50,-45])                    # yt_ocean,N
#region_lon   = np.array([141,146])                    # xt_ocean,E
#region_lon   = region_lon - 360                         # -> [-213, -205]

#region = [region_lat, region_lon]

Starting ACCESS-OM2 and SOTS T/S analysis.


In [ ]:
#----------------------------------------------------------------------------------------------
# This section collects all the experiment / degrees / variable paths
#----------------------------------------------------------------------------------------------
print(f'Processing ACCESS {var} data...')
sys.stdout.flush()
    

path_dict = {}

catalogs = []

for exp in experiments:

    if isinstance(exp, list):  # both tenth degree + extention
        merged_name = exp[0]   
        path_dict[merged_name] = []
    
        #bug here for salt vs pot_rho
        for subexp in exp:
            filtered = catalog[subexp].search(
                variable=var,
                frequency='1mon',
                realm='ocean',
            )
            df = filtered.df
            df['start_date'] = pd.to_datetime(df['start_date'])
            df['end_date']   = pd.to_datetime(df['end_date'])
    
            df = df[
                (df['start_date'] >= np.datetime64("2010-01-01")) &
                (df['end_date']   <= np.datetime64("2023-12-31"))
            ]
    
            file_list = df['path'].tolist()
            path_dict[merged_name].extend(file_list)
    
        print(f"{merged_name}: {len(path_dict[merged_name])} files")
    
    else:  
        path_dict[exp] = []
        filtered = catalog[exp].search(
            variable=var,
            frequency='1mon',
            realm='ocean',
        )
        df = filtered.df
        df['start_date'] = pd.to_datetime(df['start_date'])
        df['end_date']   = pd.to_datetime(df['end_date'])
    
        df = df[
            (df['start_date'] >= np.datetime64("2010-01-01")) &
            (df['end_date']   <= np.datetime64("2023-12-31"))
        ]
    
        file_list = df['path'].tolist()
        path_dict[exp].extend(file_list)
    
        print(f"{exp}: {len(file_list)} files")
    